# ML Trading Model Training Notebook

This notebook trains a LightGBM model for trade signal prediction.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
import json
import warnings
warnings.filterwarnings('ignore')

## Load Data

In [ ]:
# Load exported candle data
DATA_PATH = '../storage/ml_training_data/'
df = pd.read_csv(f'{DATA_PATH}binance_BTCUSDT_2024-01-01_2024-12-31.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
df = df.set_index('timestamp').sort_index()
print(f"Loaded {len(df)} candles")
df.head()

## Feature Engineering

In [ ]:
# Create triple-barrier labels
def create_labels(df, profit_threshold=0.02, loss_threshold=-0.01, hold_days=5):
    """
    Triple barrier: profit target, stop loss, or time exit
    """
    labels = []
    for i in range(len(df)):
        if i + hold_days >= len(df):
            labels.append('hold')
            continue
        
        future_prices = df['close'].iloc[i+1:i+hold_days+1]
        entry_price = df['close'].iloc[i]
        
        max_return = (future_prices.max() - entry_price) / entry_price
        min_return = (future_prices.min() - entry_price) / entry_price
        
        if max_return >= profit_threshold:
            labels.append('buy')
        elif min_return <= loss_threshold:
            labels.append('sell')
        else:
            labels.append('hold')
    
    return labels

df['label'] = create_labels(df)
print(df['label'].value_counts())

In [ ]:
# Feature list matching Ruby pipeline
FEATURE_COLS = ['open', 'high', 'low', 'close', 'volume', 'rsi', 'macd', 'volatility']

# Drop rows with NaN
df = df.dropna(subset=FEATURE_COLS)
print(f"Dataset size after dropping NaN: {len(df)}")

# Train/test split (last 20% for testing)
train_size = int(len(df) * 0.8)
train_df = df.iloc[:train_size]
test_df = df.iloc[train_size:]

X_train = train_df[FEATURE_COLS]
y_train = train_df['label'].map({'buy': 1, 'sell': -1, 'hold': 0})
X_test = test_df[FEATURE_COLS]
y_test = test_df['label'].map({'buy': 1, 'sell': -1, 'hold': 0})

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

## Train LightGBM Model

In [ ]:
params = {
    'objective': 'multiclass',
    'num_class': 3,
    'metric': 'multi_logloss',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42
}

# Create dataset
train_data = lgb.Dataset(X_train, label=y_train + 1)  # Shift labels to 0,1,2
test_data = lgb.Dataset(X_test, label=y_test + 1, reference=train_data)

# Train with early stopping
model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, test_data],
    valid_names=['train', 'valid'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

print(f"Best iteration: {model.best_iteration}")

## Walk-Forward Validation

In [ ]:
# Purged K-Fold walk-forward validation
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5, gap=10)
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    train_ds = lgb.Dataset(X_tr, label=y_tr + 1)
    val_ds = lgb.Dataset(X_val, label=y_val + 1, reference=train_ds)
    
    fold_model = lgb.train(
        params, train_ds,
        num_boost_round=500,
        valid_sets=[val_ds],
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )
    
    preds = fold_model.predict(X_val).argmax(axis=1) - 1
    accuracy = (preds == y_val).mean()
    cv_scores.append(accuracy)
    print(f"Fold {fold+1}: Accuracy = {accuracy:.4f}")

print(f"\nMean CV Accuracy: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")

## Export Model

In [ ]:
# Save model
MODEL_PATH = '../storage/ml_models/'
import os
os.makedirs(MODEL_PATH, exist_ok=True)

model.save_model(f'{MODEL_PATH}signal_model.txt')

# Save metadata
metadata = {
    'name': 'signal_model',
    'version': 1,
    'training_date': str(pd.Timestamp.today().date()),
    'feature_list': FEATURE_COLS,
    'performance_metrics': {
        'cv_accuracy': float(np.mean(cv_scores)),
        'cv_std': float(np.std(cv_scores))
    },
    'model_type': 'lightgbm'
}

with open(f'{MODEL_PATH}metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Model saved to {MODEL_PATH}")
print(json.dumps(metadata, indent=2))

## Feature Importance

In [ ]:
import matplotlib.pyplot as plt

importance = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': model.feature_importance()
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 5))
plt.barh(importance['feature'], importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()